In [4]:
import pandas as pd
import numpy as np


# 1. 원본 데이터 불러오기
df = pd.read_csv("../../data/merged_df.csv")


# 2. 이벤트 순서 정렬
event_order_map = {
    'received': 0,
    'viewed': 1,
    'completed': 2,
    'transaction': 3
}

df['event_order'] = df['event'].map(event_order_map).fillna(99)

df = df.sort_values(
    by=['person', 'offer_id', 'time', 'event_order'],
    ascending=[True, True, True, True]
).reset_index(drop=True)


# 3. received 기준 수신 순번 생성
df['is_received'] = (df['event'] == 'received').astype(int)

df['receive_seq'] = (
    df.groupby(['person', 'offer_id'])['is_received']
      .cumsum()
)


# 4. 퍼널 요약 함수
def make_funnel_summary(dataframe):
    base = (
        dataframe[(dataframe['event'] == 'received') & (dataframe['receive_seq'] > 0)]
        [['person', 'offer_id', 'receive_seq']]
        .drop_duplicates()
        .copy()
    )

    viewed = (
        dataframe[(dataframe['event'] == 'viewed') & (dataframe['receive_seq'] > 0)]
        .groupby(['person', 'offer_id', 'receive_seq'], as_index=False)
        .agg(viewed=('event', 'size'))
    )
    viewed['viewed'] = 1

    completed = (
        dataframe[(dataframe['event'] == 'completed') & (dataframe['receive_seq'] > 0)]
        .groupby(['person', 'offer_id', 'receive_seq'], as_index=False)
        .agg(completed=('event', 'size'))
    )
    completed['completed'] = 1

    summary = (
        base
        .merge(viewed, on=['person', 'offer_id', 'receive_seq'], how='left')
        .merge(completed, on=['person', 'offer_id', 'receive_seq'], how='left')
    )

    summary['viewed'] = summary['viewed'].fillna(0).astype(int)
    summary['completed'] = summary['completed'].fillna(0).astype(int)
    summary['received'] = 1

    return summary


# 5. 제거 전 퍼널 수치
before_funnel = make_funnel_summary(df)

before_received = before_funnel['received'].sum()
before_viewed = before_funnel['viewed'].sum()
before_completed = before_funnel['completed'].sum()

before_recv_view_rate = before_viewed / before_received if before_received > 0 else np.nan
before_view_comp_rate = before_completed / before_viewed if before_viewed > 0 else np.nan
before_recv_comp_rate = before_completed / before_received if before_received > 0 else np.nan


# 6. 이상 순서 수신 건 찾기
viewed_summary = (
    df[(df['event'] == 'viewed') & (df['receive_seq'] > 0)]
    .groupby(['person', 'offer_id', 'receive_seq'], as_index=False)
    .agg(first_viewed_time=('time', 'min'))
)

completed_summary = (
    df[(df['event'] == 'completed') & (df['receive_seq'] > 0)]
    .groupby(['person', 'offer_id', 'receive_seq'], as_index=False)
    .agg(first_completed_time=('time', 'min'))
)

receive_base = (
    df[(df['event'] == 'received') & (df['receive_seq'] > 0)]
    [['person', 'offer_id', 'receive_seq']]
    .drop_duplicates()
    .merge(viewed_summary, on=['person', 'offer_id', 'receive_seq'], how='left')
    .merge(completed_summary, on=['person', 'offer_id', 'receive_seq'], how='left')
)

bad_keys = receive_base[
    receive_base['first_viewed_time'].notna() &
    receive_base['first_completed_time'].notna() &
    (receive_base['first_completed_time'] < receive_base['first_viewed_time'])
][['person', 'offer_id', 'receive_seq']].copy()

print("=" * 60)
print("이상 순서 수신 건 수")
print("=" * 60)
print(len(bad_keys))


# 7. 이상 순서 viewed 제거
df['key'] = (
    df['person'].astype(str) + '|' +
    df['offer_id'].astype(str) + '|' +
    df['receive_seq'].astype(str)
)

bad_keys['key'] = (
    bad_keys['person'].astype(str) + '|' +
    bad_keys['offer_id'].astype(str) + '|' +
    bad_keys['receive_seq'].astype(str)
)

remove_mask = (
    (df['event'] == 'viewed') &
    (df['key'].isin(bad_keys['key']))
)

df_clean = df.loc[~remove_mask].copy().reset_index(drop=True)


# 8. 제거 후 퍼널 수치
after_funnel = make_funnel_summary(df_clean)

after_received = after_funnel['received'].sum()
after_viewed = after_funnel['viewed'].sum()
after_completed = after_funnel['completed'].sum()

after_recv_view_rate = after_viewed / after_received if after_received > 0 else np.nan
after_view_comp_rate = after_completed / after_viewed if after_viewed > 0 else np.nan
after_recv_comp_rate = after_completed / after_received if after_received > 0 else np.nan


# 9. 비교표
compare_df = pd.DataFrame({
    '지표': [
        'Received 건수',
        'Viewed 건수',
        'Completed 건수',
        'Received → Viewed 전환율',
        'Viewed → Completed 전환율',
        'Received → Completed 전환율'
    ],
    '제거 전': [
        before_received,
        before_viewed,
        before_completed,
        before_recv_view_rate,
        before_view_comp_rate,
        before_recv_comp_rate
    ],
    '제거 후': [
        after_received,
        after_viewed,
        after_completed,
        after_recv_view_rate,
        after_view_comp_rate,
        after_recv_comp_rate
    ]
})

compare_df['변화량'] = compare_df['제거 후'] - compare_df['제거 전']


# 10. 출력용 테이블 따로 만들기
rate_rows = [
    'Received → Viewed 전환율',
    'Viewed → Completed 전환율',
    'Received → Completed 전환율'
]

compare_display = compare_df.copy()

for col in ['제거 전', '제거 후', '변화량']:
    compare_display[col] = compare_display[col].astype(object)

    is_rate = compare_display['지표'].isin(rate_rows)
    is_count = ~is_rate

    compare_display.loc[is_rate, col] = compare_display.loc[is_rate, col].map(lambda x: f"{x:.2%}")
    compare_display.loc[is_count, col] = compare_display.loc[is_count, col].map(lambda x: f"{int(x):,}")

print("\n" + "=" * 60)
print("이상 순서 제거 전/후 퍼널 비교")
print("=" * 60)
display(compare_display)


# 11. 추가 검증
print("\n" + "=" * 60)
print("추가 검증")
print("=" * 60)
print("제거한 viewed 행 수:", remove_mask.sum())
print("제거 전 viewed 건수 - 제거 후 viewed 건수:", before_viewed - after_viewed)

이상 순서 수신 건 수
4205

이상 순서 제거 전/후 퍼널 비교


,지표,제거 전,제거 후,변화량
0,Received 건수,"76,277","76,277",0
1,Viewed 건수,"57,725","53,520","-4,205"
2,Completed 건수,"33,101","33,101",0
3,Received → Viewed 전환율,75.68%,70.17%,-5.51%
4,Viewed → Completed 전환율,57.34%,61.85%,4.51%
5,Received → Completed 전환율,43.40%,43.40%,0.00%



추가 검증
제거한 viewed 행 수: 4205
제거 전 viewed 건수 - 제거 후 viewed 건수: 4205
